# 02 · Bronze → Silver
Reads the bronze Delta tables and writes typed, cleaned and deduplicated Delta tables to the `silver` schema:
* casts keys and quantities to integers, parses currency text (`$2,024.99`) and long dates (`Friday, August 25, 2017`)
* trims strings and turns empty strings into NULL
* gives columns business-friendly names (e.g. `Product` → `ProductName`)
* keeps the latest row per primary key (see `DATASETS` in the config)

In [0]:
%run ../../config/pipeline_config

In [0]:
%run ../../utils/common_functions

In [0]:
META = ["ingestion_timestamp", "source_file_name"]

def transform_product(df):
    return df.select(to_int("ProductKey").alias("ProductKey"), clean_string("Product").alias("ProductName"),
                     parse_money("StandardCost").alias("StandardCost"), clean_string("Color").alias("Color"),
                     clean_string("Subcategory").alias("Subcategory"), clean_string("Category").alias("Category"),
                     clean_string("BackgroundColorFormat").alias("BackgroundColorFormat"),
                     clean_string("FontColorFormat").alias("FontColorFormat"), *META)

def transform_region(df):
    return df.select(to_int("SalesTerritoryKey").alias("SalesTerritoryKey"), clean_string("Region").alias("Region"),
                     clean_string("Country").alias("Country"), clean_string("Group").alias("Group"), *META)

def transform_reseller(df):
    return df.select(to_int("ResellerKey").alias("ResellerKey"), clean_string("BusinessType").alias("BusinessType"),
                     clean_string("Reseller").alias("ResellerName"), clean_string("City").alias("City"),
                     clean_string("StateProvince").alias("StateProvince"), clean_string("CountryRegion").alias("CountryRegion"), *META)

def transform_sales(df):
    return df.select(clean_string("SalesOrderNumber").alias("SalesOrderNumber"), parse_long_date("OrderDate").alias("OrderDate"),
                     to_int("ProductKey").alias("ProductKey"), to_int("ResellerKey").alias("ResellerKey"),
                     to_int("EmployeeKey").alias("EmployeeKey"), to_int("SalesTerritoryKey").alias("SalesTerritoryKey"),
                     to_int("Quantity").alias("Quantity"), parse_money("UnitPrice").alias("UnitPrice"),
                     parse_money("Sales").alias("Sales"), parse_money("Cost").alias("Cost"), *META)

def transform_salesperson(df):
    return df.select(to_int("EmployeeKey").alias("EmployeeKey"), clean_string("EmployeeID").alias("EmployeeID"),
                     clean_string("Salesperson").alias("SalespersonName"), clean_string("Title").alias("Title"),
                     F.lower(clean_string("UPN")).alias("UPN"), *META)

def transform_salesperson_region(df):
    return df.select(to_int("EmployeeKey").alias("EmployeeKey"), to_int("SalesTerritoryKey").alias("SalesTerritoryKey"), *META)

def transform_targets(df):
    return df.select(clean_string("EmployeeID").alias("EmployeeID"), parse_money("Target").alias("Target"),
                     parse_long_date("TargetMonth").alias("TargetMonth"), *META)

TRANSFORMS = {
    "product": transform_product, "region": transform_region, "reseller": transform_reseller,
    "sales": transform_sales, "salesperson": transform_salesperson,
    "salesperson_region": transform_salesperson_region, "targets": transform_targets,
}

In [0]:
load_summary = []

for dataset in SELECTED_DATASETS:
    cfg = DATASETS[dataset]
    bronze_df = spark.table(f"{BRONZE_SCHEMA}.{dataset}")

    silver_df = (deduplicate(TRANSFORMS[dataset](bronze_df), cfg["primary_key"])
                 .withColumn("silver_load_timestamp", F.current_timestamp()))

    silver_rows = write_delta_table(silver_df, f"{SILVER_SCHEMA}.{dataset}", f"{SILVER_PATH}{cfg['folder']}")
    bronze_rows = bronze_df.count()
    load_summary.append((dataset, bronze_rows, silver_rows, bronze_rows - silver_rows))

display(spark.createDataFrame(load_summary, "dataset STRING, bronze_rows LONG, silver_rows LONG, duplicates_removed LONG"))